# 🎬 Wan 2.1 AI Video Generator

Generate high-quality AI videos using Alibaba's Wan 2.1 model.

**Requirements:**
- GPU Runtime (T4 free tier works!)
- ~15GB VRAM for 1.3B model

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## 1️⃣ Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi
import torch
print(f"\n✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✅ VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q torch>=2.4.0 torchvision
!pip install -q diffusers transformers accelerate
!pip install -q imageio imageio-ffmpeg
!pip install -q huggingface_hub
print("✅ Dependencies installed!")

In [ ]:
# Clone Wan 2.1 repository
!git clone https://github.com/Wan-Video/Wan2.1.git
%cd Wan2.1
!pip install -q -r requirements.txt
print("✅ Wan 2.1 installed!")

## 2️⃣ Download Model (1.3B - fits in T4 GPU)

In [ ]:
# Download the 1.3B Text-to-Video model (smaller, fits in 16GB VRAM)
from huggingface_hub import snapshot_download

model_path = snapshot_download(
    repo_id="Wan-AI/Wan2.1-T2V-1.3B",
    local_dir="./Wan2.1-T2V-1.3B",
    local_dir_use_symlinks=False
)
print(f"✅ Model downloaded to: {model_path}")

## 3️⃣ Generate Video from Text

In [ ]:
# @title Text-to-Video Generation { run: "auto" }
PROMPT = "A serene ocean sunset with gentle waves, golden hour lighting, cinematic"  # @param {type:"string"}
NEGATIVE_PROMPT = "blurry, low quality, distorted"  # @param {type:"string"}
NUM_FRAMES = 81  # @param {type:"slider", min:17, max:81, step:8}
HEIGHT = 480  # @param [480, 720]
WIDTH = 832  # @param [832, 1280]
GUIDANCE_SCALE = 5.0  # @param {type:"slider", min:1.0, max:10.0, step:0.5}
NUM_INFERENCE_STEPS = 50  # @param {type:"slider", min:20, max:100, step:5}

In [ ]:
import torch
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video
import gc

# Clear GPU memory
gc.collect()
torch.cuda.empty_cache()

print("🔄 Loading model...")

# Load VAE with memory optimizations
vae = AutoencoderKLWan.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B",
    subfolder="vae",
    torch_dtype=torch.float16
)

# Load pipeline
pipe = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B",
    vae=vae,
    torch_dtype=torch.float16
)

# Enable memory optimizations
pipe.enable_model_cpu_offload()
pipe.enable_vae_slicing()

print("✅ Model loaded!")

In [ ]:
# Generate video
print(f"🎬 Generating video: '{PROMPT[:50]}...'")
print(f"   Frames: {NUM_FRAMES}, Resolution: {WIDTH}x{HEIGHT}")

output = pipe(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    height=HEIGHT,
    width=WIDTH,
    num_frames=NUM_FRAMES,
    guidance_scale=GUIDANCE_SCALE,
    num_inference_steps=NUM_INFERENCE_STEPS,
)

# Save video
output_path = "wan_output.mp4"
export_to_video(output.frames[0], output_path, fps=16)
print(f"✅ Video saved: {output_path}")

In [ ]:
# Play the video
from IPython.display import Video
Video(output_path, embed=True, width=640)

In [ ]:
# Download video
from google.colab import files
files.download(output_path)

## 4️⃣ Image-to-Video (Optional)

Generate video from an image prompt.

In [ ]:
# Upload an image for Image-to-Video
from google.colab import files
from PIL import Image
import io

print("📤 Upload an image:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    input_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
    input_image = input_image.resize((832, 480))
    display(input_image)
    print(f"✅ Image loaded: {filename}")

In [ ]:
# Download I2V model and generate
from huggingface_hub import snapshot_download
from diffusers import WanImageToVideoPipeline
import gc

# Clear previous model
del pipe, vae
gc.collect()
torch.cuda.empty_cache()

print("🔄 Loading Image-to-Video model...")

i2v_pipe = WanImageToVideoPipeline.from_pretrained(
    "Wan-AI/Wan2.1-I2V-14B-480P",
    torch_dtype=torch.float16
)
i2v_pipe.enable_model_cpu_offload()
i2v_pipe.enable_vae_slicing()

print("✅ I2V Model loaded!")

In [ ]:
# @title Generate Video from Image
I2V_PROMPT = "The scene comes alive with gentle movement"  # @param {type:"string"}

print(f"🎬 Generating video from image...")

i2v_output = i2v_pipe(
    image=input_image,
    prompt=I2V_PROMPT,
    num_frames=81,
    guidance_scale=5.0,
    num_inference_steps=50,
)

i2v_output_path = "wan_i2v_output.mp4"
export_to_video(i2v_output.frames[0], i2v_output_path, fps=16)
print(f"✅ Video saved: {i2v_output_path}")

In [ ]:
# Play and download
from IPython.display import Video
display(Video(i2v_output_path, embed=True, width=640))
files.download(i2v_output_path)

## 5️⃣ Batch Generation

In [ ]:
# Generate multiple videos from different prompts
BATCH_PROMPTS = [
    "A futuristic city at night with neon lights and flying cars",
    "A peaceful forest with sunlight filtering through trees",
    "Ocean waves crashing on a tropical beach at sunset"
]

# Reload T2V model if needed
if 'pipe' not in dir():
    gc.collect()
    torch.cuda.empty_cache()
    
    vae = AutoencoderKLWan.from_pretrained(
        "Wan-AI/Wan2.1-T2V-1.3B",
        subfolder="vae",
        torch_dtype=torch.float16
    )
    pipe = WanPipeline.from_pretrained(
        "Wan-AI/Wan2.1-T2V-1.3B",
        vae=vae,
        torch_dtype=torch.float16
    )
    pipe.enable_model_cpu_offload()
    pipe.enable_vae_slicing()

import os
os.makedirs("batch_outputs", exist_ok=True)

for i, prompt in enumerate(BATCH_PROMPTS):
    print(f"\n🎬 [{i+1}/{len(BATCH_PROMPTS)}] Generating: {prompt[:40]}...")
    
    output = pipe(
        prompt=prompt,
        negative_prompt="blurry, low quality",
        height=480,
        width=832,
        num_frames=49,  # Shorter for batch
        guidance_scale=5.0,
        num_inference_steps=40,
    )
    
    output_path = f"batch_outputs/video_{i+1}.mp4"
    export_to_video(output.frames[0], output_path, fps=16)
    print(f"   ✅ Saved: {output_path}")

print("\n✅ Batch generation complete!")

In [ ]:
# Download all batch videos as zip
!zip -r batch_videos.zip batch_outputs/
files.download('batch_videos.zip')

---
## 📚 Resources

- [Wan 2.1 GitHub](https://github.com/Wan-Video/Wan2.1)
- [Wan 2.1 on Hugging Face](https://huggingface.co/Wan-AI)
- [Diffusers Documentation](https://huggingface.co/docs/diffusers/main/en/api/pipelines/wan)

**Tips:**
- Use Colab Pro for A100 GPU (40GB VRAM) to run 14B model
- Lower `num_frames` if running out of memory
- Add detailed prompts for better results